# 01 · Análisis exploratorio

Objetivo: entender el target, su comportamiento en el tiempo y el poder predictivo de las variables **de originación** antes de modelar.

Todo lo que se calcula sobre variables (IV, correlaciones) usa **solo el conjunto de entrenamiento**, para que ninguna decisión de modelado mire el test.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from lcrisk import config as C
from lcrisk.data import load_processed
from lcrisk.features import engineer, get_xy
from lcrisk.selection import iv_summary, woe_iv_table, correlated_pairs, cramers_v

pd.set_option("display.max_columns", 80)
sns.set_theme(style="whitegrid")

df = load_processed()          # requiere: python scripts/make_dataset.py
print(df.shape)
df.head()

## 1. Target y cosechas

La tasa de default cambia con el ciclo económico y con la política de admisión de LendingClub. Un split aleatorio esconde esto; el split temporal lo expone.

In [ ]:
vintage = df.assign(year=df[C.DATE_COL].dt.year).groupby(["year", "split"], observed=True)[C.TARGET].agg(["size", "mean"])
vintage.columns = ["préstamos", "tasa_default"]
vintage["tasa_default"] = (vintage["tasa_default"] * 100).round(2)
vintage

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
q = df.assign(q=df[C.DATE_COL].dt.to_period("Q")).groupby("q")[C.TARGET].mean() * 100
q.plot(ax=ax, marker="o")
for s, color in zip(["train", "valid", "test"], ["#4C72B0", "#DD8452", "#55A868"]):
    sub = df[df.split == s][C.DATE_COL]
    ax.axvspan(sub.min().to_period("Q").ordinal, sub.max().to_period("Q").ordinal, alpha=0.12, color=color, label=s)
ax.set(ylabel="Tasa de default (%)", xlabel="Trimestre de originación", title="Tasa de default por cosecha")
ax.legend(); plt.show()

## 2. Faltantes por año

LendingClub incorporó variables de buró en 2012 aprox. El patrón de faltantes es *informativo* (indica la época), no aleatorio. Por eso el pipeline agrega indicadores de faltante en vez de imputar y olvidar.

In [ ]:
miss = df.assign(year=df[C.DATE_COL].dt.year).groupby("year")[C.CREDIT_BUREAU].apply(lambda g: g.isna().mean()).T * 100
plt.figure(figsize=(10, 12))
sns.heatmap(miss, cmap="Reds", cbar_kws={"label": "% faltante"})
plt.title("% de valores faltantes por variable y año de originación"); plt.show()

## 3. Information Value (solo train)

In [ ]:
dfe = engineer(df)
X, y, num, cat = get_xy(dfe)
tr = dfe["split"] == "train"
iv = iv_summary(X[tr], y[tr])
iv.head(30)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
top = iv.head(30).iloc[::-1]
ax.barh(top["feature"], top["iv"], color="#4C72B0")
for x_, label in [(0.02, "débil"), (0.10, "medio"), (0.30, "fuerte")]:
    ax.axvline(x_, ls="--", color="grey", lw=0.8); ax.text(x_, len(top) - 0.5, label, fontsize=8, color="grey")
ax.set(title="Information Value por variable (train)", xlabel="IV"); plt.show()

### WoE de las variables más fuertes

El WoE debe ser **monótono** en variables de riesgo (más tasa → más default). Si no lo es, suele indicar un problema de datos o una relación no lineal que un modelo de árboles capturará mejor.

In [ ]:
for col in iv["feature"].head(4):
    tab, iv_val = woe_iv_table(X.loc[tr, col], y[tr])
    fig, ax1 = plt.subplots(figsize=(9, 3.5))
    ax1.bar(range(len(tab)), tab["bad_rate"] * 100, color="#DD8452", alpha=0.7, label="tasa de default (%)")
    ax2 = ax1.twinx(); ax2.plot(range(len(tab)), tab["woe"], marker="o", color="#4C72B0", label="WoE")
    ax1.set_xticks(range(len(tab))); ax1.set_xticklabels([str(i) for i in tab.index], rotation=45, ha="right", fontsize=8)
    ax1.set_title(f"{col} — IV = {iv_val:.3f}"); ax1.legend(loc="upper left"); ax2.legend(loc="upper right")
    plt.tight_layout(); plt.show()

## 4. Redundancia entre variables

Los modelos de árboles toleran colinealidad; la regresión logística no. Aquí se documenta qué pares son casi equivalentes (Spearman ≥ 0.8) por si se quiere un modelo lineal más parsimonioso.

In [ ]:
correlated_pairs(X.loc[tr, num], threshold=0.8)

In [ ]:
# grade y sub_grade son jerárquicas (V de Cramér ≈ 1): por eso sub_grade entra como ordinal y no como one-hot junto a grade
print("Cramér's V grade vs sub_grade:", round(cramers_v(df.loc[tr, 'grade'], df.loc[tr, 'sub_grade']), 3))